In [ ]:
# Databricks Notebook: Gold Summary for Futures Positions Dashboard
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

CATALOG = "demo_catalog"
SCHEMA  = "demo_schema"
SILVER  = f"{CATALOG}.{SCHEMA}.silver_futures_positions"
GOLD    = f"{CATALOG}.{SCHEMA}.gold_futures_positions_summary"

# ===== Spark / Delta Performance Configuration =====
# optimizeWrite: 커밋 전 소파일을 병합 → 소파일 누적 방지
spark.conf.set("spark.databricks.delta.optimizeWrite", "true")
# autoCompact: 쓰기 완료 후 백그라운드 컴팩션 자동 트리거
spark.conf.set("spark.databricks.delta.autoCompact", "true")
# AQE: Window 함수(row_number) 및 MERGE에서 런타임 통계 기반 동적 재플래닝
spark.conf.set("spark.sql.adaptive.enabled", "true")
# 셔플 후 소규모/빈 파티션을 동적으로 병합하여 Task 오버헤드 감소
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
# 심볼별 데이터 Skew(BTCUSDT 편중) 자동 감지 및 처리
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")

# =========================
# (A) Gold 테이블 생성
# =========================
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {GOLD} (
  uid        STRING,
  symbol     STRING,
  entryPrice DOUBLE,
  markPrice  DOUBLE,
  pnl        DOUBLE,
  roe        DOUBLE,
  amount     DOUBLE,
  leverage   DOUBLE,
  event_time TIMESTAMP,
  updated_at TIMESTAMP
) USING DELTA
TBLPROPERTIES (
  'delta.logRetentionDuration'         = 'interval 30 days',
  'delta.deletedFileRetentionDuration' = 'interval 14 days'
)
""")

# =========================
# (B) 최신 포지션만 추출
# =========================
silver = spark.table(SILVER)

w = Window.partitionBy("uid","symbol").orderBy(F.col("event_time").desc())

latest_positions = (
    silver
      .withColumn("rn", F.row_number().over(w))
      .where("rn = 1")
      .drop("rn")
      .withColumn("updated_at", F.current_timestamp())
)

# =========================
# (C) Delta MERGE (idempotent upsert)
# - overwrite 대신 MERGE를 사용하여 멱등성 보장
# - 재실행/백필 시 중복 없이 최신 포지션으로 원자적 갱신
# =========================
target = DeltaTable.forName(spark, GOLD)
(target.alias("t")
  .merge(
      latest_positions.alias("s"),
      "t.uid = s.uid AND t.symbol = s.symbol"
  )
  .whenMatchedUpdate(set={
      "entryPrice":  "s.entryPrice",
      "markPrice":   "s.markPrice",
      "pnl":         "s.pnl",
      "roe":         "s.roe",
      "amount":      "s.amount",
      "leverage":    "s.leverage",
      "event_time":  "s.event_time",
      "updated_at":  "s.updated_at"
  })
  .whenNotMatchedInsertAll()
  .execute()
)

print(f"[SUMMARY] MERGE complete: {GOLD}")